In [9]:
import requests

url = "https://securenvoy.com/wp-content/uploads/2018/08/SecurEnvoy-RESTful-API-Guide.pdf"
filename = "SecurEnvoy-RESTful-API-Guide.pdf"

response = requests.get(url, timeout=30)
response.raise_for_status()

with open(filename, "wb") as f:
    f.write(response.content)

print(f"Downloaded: {filename}")

Downloaded: SecurEnvoy-RESTful-API-Guide.pdf


In [10]:
# ---- Platform Credentials ---------------------------------------------------
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # reads GOOGLE_API_KEY and ENDPOINT_URL from .env

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
ENDPOINT_URL   = os.getenv("ENDPOINT_URL")
GEMINI_EMBEDDING_MODEL = "models/gemini-embedding-2"

# ---- Configure the Gemini client --------------------------------------------
# The new google-genai SDK has no genai.configure(); create a Client instead
# and point it at the lab's proxy with base_url.
client = genai.Client(
    api_key=GOOGLE_API_KEY,
    http_options=types.HttpOptions(base_url=ENDPOINT_URL),
)

print(f"Endpoint : {ENDPOINT_URL}")
print(f"API key  : {'configured' if GOOGLE_API_KEY else 'MISSING'}")

# Quick check that the client can reach the proxy.
test = client.models.embed_content(model=GEMINI_EMBEDDING_MODEL, contents="hello")
print(f"Embedding check: OK ({len(test.embeddings[0].values)} dimensions)")


Endpoint : https://api.enterprisesi.co/api/v1/genai
API key  : configured
Embedding check: OK (3072 dimensions)


In [11]:
from langchain_core.embeddings import Embeddings
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=GOOGLE_API_KEY,
    base_url=ENDPOINT_URL,
)

_embedding_client = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY,
    base_url=ENDPOINT_URL,
)


from langchain_core.vectorstores import InMemoryVectorStore

class CustomGeminiEmbeddings(Embeddings):
    def __init__(self, model=None):
        self.model = GEMINI_EMBEDDING_MODEL

    def embed_documents(self, texts):
        return [self.embed_query(text) for text in texts]

    def embed_query(self, text):
        response = genai.embed_content(model=self.model, content=text)
        return response["embedding"]

embeddings = CustomGeminiEmbeddings()
vectorstore = InMemoryVectorStore(embedding=embeddings)

def extract_text(response) -> str:
    """Normalise a ChatGoogleGenerativeAI response into a plain string."""
    content = response.content
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(part.get("text", "") for part in content if isinstance(part, dict))
    return str(content)

In [12]:
from typing import TypedDict

class GraphState(TypedDict):
    requirement:      str   # raw requirement text entered by the user
    retrieved_chunks: str   # relevant PDF sections retrieved by the Planner
    plan:             str   # numbered implementation plan - written by Planner
    code:             str   # Python code - written by Coder
    review:           str   # structured code review - written by Reviewer

In [13]:
import os
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

PDF_PATH = "SecurEnvoy-RESTful-API-Guide.pdf"


def load_requirements(pdf_path: str) -> list:
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"Could not find '{pdf_path}'.")

    reader = PdfReader(pdf_path)
    pages = [
        Document(page_content=page.extract_text() or "", metadata={"page": i + 1})
        for i, page in enumerate(reader.pages)
    ]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=600, chunk_overlap=100,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = splitter.split_documents(pages)
    print(f"Loaded {len(pages)} pages → {len(chunks)} chunks")
    return chunks


def build_index(chunks: list):
    vectorstore = InMemoryVectorStore(embedding=embeddings)
    vectorstore.add_documents(chunks)
    print("In-memory vector index ready.")
    return vectorstore.as_retriever(search_kwargs={"k": 4})


chunks    = load_requirements(PDF_PATH)
retriever = build_index(chunks)

Loaded 17 pages → 52 chunks


AttributeError: module 'google.genai' has no attribute 'embed_content'